# 2030 – Model Encoding, Feature Selection, modeling and valuation for each brand

## Notebook Overview
This notebook shows our approach of spliting into brand specific models.

- Inputs: `11_processed_train_data.csv`, `11_processed_test_data.csv`.
- Outputs: `30_submission_YYYYMMDD_HHMM_rf.csv`, `30_submission_YYYYMMDD_HHMM_etr.csv`, `30_submission_YYYYMMDD_HHMM_hgbr.csv` and `30_submission_YYYYMMDD_HHMM_hybrid.csv` in `../data/submissions/`.

### Preprocessing
This notebook first splits the data into validation and training stratified by brand (important for later). It then imputes the categorical missing values. After that it splits the data set into different brands to from now on do steps only for the specific brand. In the next step we impute the numerical missing values. After that we both one hot and target encode the categorical features. We decide not to scale our data, since it is useless for the models we use here.

### Feature Selection
We use filter (variance, correlation) and wrapper (RFE) methods, selecting features based on validation performance while avoiding leakage.

### Models
For the Models we use a Random Forest Regressor, an Extra Tree Regressor and a Hist Gradient Boosting Regressor. For all of them we use parameter grid search. At the end we also combine Extra Tree Regressor and a Hist Gradient Boosting Regressor (with weights), which is our best model we got in this project.

### Evaluation

TBD

Note: All model selection uses the validation split; only the final chosen model is trained on train+val before generating test predictions.

# Table of Contents

<a id="top"></a>

- [Notebook Overview](#notebook-overview)
- [1. Import Libraries and functions](#sec-1-imports)
- [2. Load Dataset](#sec-2-load)
- [3. Imputation Strategies](#sec-3-sanity)
- [Intermediate step: Split Brands](#intermediate-step-split-brands)
- [3.2 Numerical Imputation](#3-2-numerical-imputation)
- [5. Feature Engineering](#5-feature-engineering)
- [6. Scaling](#6-scaling)
- [7. Encoding](#7-encoding)
- [8. Feature Selection](#sec-4-filter)
- [9. Models](#9-models)
- [9.2 Extra Trees Regressor](#9-2-extra-trees-regressor)
- [9.3 Hist Gradient Boosting Regressor](#9-3-hist-gradient-boosting-regressor)
- [9.4 Hybrid Model (Extra Tree Regressor * 0,6 + High Gradient Boosting * 0,4)](#9-4-hybrid-model-extra-tree-regressor-0-6-high-gradient-boosting-0-4)


<a id="sec-1-imports"></a>
## 1. Import Libraries and functions


All needed imports sorted by library.

In [1]:
# Standard library
import json
import math
import os
import re
import warnings
from datetime import datetime
from pathlib import Path

# third-party
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv

# sklearn: model selection / metrics
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid, train_test_split

# sklearn: preprocessing / pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, RobustScaler, TargetEncoder

# sklearn: models
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

# sklearn: feature selection / base / inspection
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.inspection import permutation_importance

# sklearn: evaluation
from sklearn.metrics import mean_absolute_error, median_absolute_error, mean_squared_error, r2_score

In [2]:
# Function to summarize missing values for a given dataset
def missing_report(X: pd.DataFrame, name: str) -> pd.DataFrame:
    """Generate and display a simple missing-value report for a DataFrame.

    X : pd.DataFrame
        Input data for which missing values should be summarized.
    name : str
        A label used in printed output to identify this dataset (e.g. 'train', 'test').

    Returns
    pd.DataFrame
        A DataFrame with two columns:
        - 'n_missing': absolute number of missing values per column
        - 'pct_missing': percentage of missing values per column

        If no missing values are found, an empty DataFrame with the same columns is returned.
    """
    # Count missing values per column
    mv = X.isna().sum()
    # Keep only columns with at least one missing value, sorted by count
    mv = mv[mv > 0].sort_values(ascending=False)

    # If there are no missing values, print a short message and return an empty report
    if mv.empty:
        print(f"[{name}] No missing values found. (n_rows={len(X)})")
        return pd.DataFrame(columns=["n_missing", "pct_missing"])

    # Compute percentage of missing values per column (rounded to two decimals)
    pct = (mv / len(X) * 100).round(2)

    # Combine counts and percentages into a single DataFrame
    report = pd.DataFrame({"n_missing": mv, "pct_missing": pct})

    # Print and display the report for quick inspection
    print(f"[{name}] Missing Values (n_rows={len(X)}):")
    display(report)

    return report

In [3]:
# Function to build an imputation pipeline template based on available features
def make_template_for(features):
    """Create an imputation pipeline tailored to the given feature set.

    features : list-like
        Collection of feature names present in the dataset.

    Returns
    sklearn.Pipeline
        An imputation pipeline configured with:
        - low-cardinality categorical columns (Brand, fuelType, transmission), if present
        - high-cardinality categorical columns (model), if present
    """
    # Select low-cardinality categorical columns that are actually present
    low_card = [c for c in ["Brand", "fuelType", "transmission"] if c in features]
    # Select high-cardinality categorical columns that are actually present
    high_card = [c for c in ["model"] if c in features]

    # Create a pipeline that knows how to impute low- and high-card categorical features
    return create_imputation_pipeline(low_card_cols=low_card, high_card_cols=high_card)

In [4]:
# Function to apply all fitted imputers in a consistent order
def apply_imputers(X):
    """Apply all pre-fitted imputers to the input data `X` in sequence.

    This function assumes that the global imputers (trans_imputer, fuel_imputer,
    brand_imputer, model_imputer) have already been fitted on the training data.
    It returns the transformed feature matrix with missing values imputed.
    """
    # Apply transmission imputer
    X = trans_imputer.transform(X)
    # Apply fuel type imputer
    X = fuel_imputer.transform(X)
    # Apply brand imputer
    X = brand_imputer.transform(X)
    # Apply model imputer
    X = model_imputer.transform(X)
    return X

In [5]:
# Function to drop low-variance features (below a given threshold)
def drop_low_variance_features(x, threshold):
    """Remove columns whose variance is below `threshold`.
    Returns the reduced frame and the list of dropped features for consistency across splits.
    """
    # Calculate the variance of each feature
    variances = x.var()
    # Identify features with variance below the threshold
    low_variance_features = variances[variances < threshold].index
    # Drop low-variance features
    x = x.drop(columns=low_variance_features)
    return x, low_variance_features

<a id="sec-2-load"></a>
## 2. Load Dataset


Load the mapped and normalized datasets produced in notebook 11 and confirm shapes before proceeding. <br>

In [6]:
# Load the data paths
data_dir = "../data/"

# Load automatically the latest folder based on date pattern
latest_folder = sorted(os.listdir(os.path.join(data_dir, "processed_data")))[-1]

# Load the raw data into a pandas dataframe
df = pd.read_csv(os.path.join(data_dir, f"processed_data/{latest_folder}/11_processed_train_data.csv"))
x_test = pd.read_csv(os.path.join(data_dir, f"processed_data/{latest_folder}/11_processed_test_data.csv"))

# put carID as Index
df.set_index("carID", inplace=True)
x_test.set_index("carID", inplace=True)

print("Loaded shape:", df.shape)
display(df.head(3))

print("Loaded test shape:", x_test.shape)
display(x_test.head(3))

Loaded shape: (74467, 12)


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,previousOwners,hasDamage
carID,,,,,,,,,,,,
69512,Volkswagen,Golf,2016.0,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,4.0,0.0
53000,Toyota,Yaris,2019.0,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,1.0,0.0
6366,Audi,Q2,2019.0,24990.0,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,4.0,0.0


Loaded test shape: (32567, 11)


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,previousOwners,hasDamage
carID,,,,,,,,,,,
89856,Hyundai,i30,NaN,Automatic,30700.0,Petrol,205.0,41.5,1.6,3.0,0.0
106581,Volkswagen,Tiguan,2017.0,Semi-Auto,NaN,Petrol,150.0,38.2,2.0,2.0,0.0
80886,BMW,2 Series,2016.0,Automatic,36792.0,Petrol,125.0,51.4,1.5,2.0,0.0


In the next cell we split our data into train and validation data. We use an 85/15 train–validation split, stratified by Brand.
Because the competition already holds out 20% of the full dataset as test data, we wanted to keep the validation size small so that we don't lose too much training data.

In [7]:
# Separate features and target from the training data
x = df.drop(columns=["price"])
y = df["price"]

# Split df (80% of total data) into training and validation
x = df.drop(columns=["price"])
y = df["price"]

# only for stratify
brand_strat = x["Brand"].fillna("Unknown")   # or "UNK"

x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.15,
    random_state=42,
    stratify=brand_strat # it trys to split the brands equaly --> good so that we have around the same percentage splits into val and train for each brand
)

print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_val shape:   {x_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"x_test shape:  {x_test.shape}")

x_train shape: (63296, 11)
y_train shape: (63296,)
x_val shape:   (11171, 11)
y_val shape:   (11171,)
x_test shape:  (32567, 11)


In the next cell we check what missing values for which type of variables we will have to impute.

In [8]:
# Reports for the splits
missing_report(x_train, "x_train")
missing_report(x_val,   "x_val")
missing_report(x_test,  "x_test")

# check which columns are categorical and which numerical
numeric_cols = x_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = x_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"#numeric_cols: {len(numeric_cols)} → {numeric_cols[:10]}{' ...' if len(numeric_cols) > 10 else ''}")
print(f"#categorical_cols: {len(categorical_cols)} → {categorical_cols[:10]}{' ...' if len(categorical_cols) > 10 else ''}")

[x_train] Missing Values (n_rows=63296):


,n_missing,pct_missing
mpg,7653,12.09
tax,6892,10.89
transmission,1887,2.98
engineSize,1742,2.75
previousOwners,1596,2.52
mileage,1525,2.41
model,1445,2.28
hasDamage,1274,2.01
fuelType,1249,1.97
year,310,0.49


[x_val] Missing Values (n_rows=11171):


,n_missing,pct_missing
mpg,1351,12.09
tax,1216,10.89
transmission,331,2.96
engineSize,313,2.80
previousOwners,292,2.61
mileage,276,2.47
model,259,2.32
hasDamage,247,2.21
fuelType,230,2.06
year,48,0.43


[x_test] Missing Values (n_rows=32567):


,n_missing,pct_missing
mpg,3840,11.79
tax,3469,10.65
year,1007,3.09
transmission,968,2.97
previousOwners,936,2.87
engineSize,875,2.69
mileage,859,2.64
model,751,2.31
fuelType,656,2.01
hasDamage,597,1.83


#numeric_cols: 7 → ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'hasDamage']
#categorical_cols: 4 → ['Brand', 'model', 'transmission', 'fuelType']


<a id="sec-3-sanity"></a>
## 3. Imputation Strategies

We will handle missing values using specific strategies for categorical and numerical features. Inbetween we split our data set into the different brands, more about that later.
We define custom imputers that can be integrated into a pipeline.

### 3.1 Categorical Imputation
Strategies for: `transmission`, `fuelType`, `Brand`, `model`.
These imputers use a combination of rule-based logic (e.g., mode per model) and Random Forrest to fill missing values.

In [9]:
# Used for low-cardinality categorical features
low_cardinality_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Used for high-cardinality categorical features like model
high_cardinality_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Helper function to create specific pipelines for each imputer
# This ensures we only try to encode columns that are actually present as features
def create_imputation_pipeline(low_card_cols, high_card_cols=[]):
    transformers = []
    if low_card_cols:
        transformers.append(('low_card', low_cardinality_pipeline, low_card_cols))
    if high_card_cols:
        transformers.append(('high_card', high_cardinality_pipeline, high_card_cols))
        
    # remainder='passthrough' keeps the numerical columns
    encoder = ColumnTransformer(transformers, remainder='passthrough', verbose_feature_names_out=False)
    
    return Pipeline([
        ('encoder', encoder),
        ("rf_model", RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
    ])
#mputes missing transmission values
class TransmissionImputer(BaseEstimator, TransformerMixin):
    
    def __init__(self, pipeline_template, min_model_count=10):
        #Preconfigured scikit-learn pipeline used to train the ML-based imputer.
        self.pipeline_template = pipeline_template
        # Minimum number of observations required for a car model to be considered reliable for rule-based imputation
        self.min_model_count = min_model_count

    def fit(self,
            X, #Input DataFrame containing features and the transmission column.
            y=None #Included for compatibility with scikit-learn transformers.
           ):
        df = X.copy()

        # store lookup tables learned from training data
        counts = df.dropna(subset=['transmission'])['model'].value_counts()
        # select models that appears more than 10 times
        self.valid_models_ = counts[counts >= self.min_model_count].index
        #group the models with the most_common_transmission 
        self.model_modes_ = (
            df.dropna(subset=['model', 'transmission'])
              .groupby('model')['transmission']
              .agg(lambda s: s.mode().iloc[0])
        )

        # Train ML model for remaining missin g values
        train_df = df[df['transmission'].notna()].copy()

        features = [
            "Brand", "model", "fuelType",
            "engineSize", "year", "mpg", "tax",
        ]

        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['transmission'])

        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # rule-based fill
        for model in self.valid_models_:
            mask = (df['model'] == model) & (df['transmission'].isna())
            #fill the transmission missing values with the most frequent transmission at the respective car model
            if model in self.model_modes_:
                df.loc[mask, 'transmission'] = self.model_modes_[model]

        # ML fallback
        missing_mask = df['transmission'].isna()
         #predict the missing transmission values with the Random Forest classifier
        if missing_mask.any():
            df.loc[missing_mask, 'transmission'] = self.ml_model_.predict(
                df.loc[missing_mask, self.features_]
            )
        return df

class FuelTypeImputer(BaseEstimator, TransformerMixin):
    
    def __init__(self, pipeline_template, min_model_count=10):
        # Preconfigured ML pipeline used for fuelType prediction
        self.pipeline_template = pipeline_template
        # Minimum number of observations required to trust rule-based imputation
        self.min_model_count = min_model_count

    def fit(self, X, y=None):
        df = X.copy()

        # Count how many times each model appears with a known fuelType
        counts = df.dropna(subset=['fuelType'])['model'].value_counts()

        # Select models with enough observations to trust rule-based imputation
        self.valid_models_ = counts[counts >= self.min_model_count].index

        # Compute the most frequent fuelType for each model
        self.model_modes_ = (
            df.dropna(subset=['model', 'fuelType'])
              .groupby('model')['fuelType']
              .agg(lambda s: s.mode().iloc[0])
        )

        # Features used by the ML fallback model
        features = [
            'Brand', 'model', 'transmission',
            'engineSize', 'year', 'mpg'
        ]

        # Training data where fuelType is known
        train_df = df[df['fuelType'].notna()]

        # Clone and train the ML model
        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['fuelType'])

        # Store trained model and feature list
        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        #  Rule-based imputation
        # Fill missing fuelType values using model-level modes
        for model in self.valid_models_:
            mask = (df['model'] == model) & (df['fuelType'].isna())
            df.loc[mask, 'fuelType'] = self.model_modes_.get(model, np.nan)

        #  ML fallback
        # Predict remaining missing fuelType values using ML model
        missing = df['fuelType'].isna()
        if missing.any():
            df.loc[missing, 'fuelType'] = self.ml_model_.predict(
                df.loc[missing, self.features_]
            )
        return df

class BrandImputer(BaseEstimator, TransformerMixin):
    
    def __init__(self, pipeline_template, min_model_count=20):
        # Preconfigured ML pipeline for brand prediction
        self.pipeline_template = pipeline_template
        # Minimum count threshold (kept higher due to brand importance)
        self.min_model_count = min_model_count

    def fit(self, X, y=None):
        df = X.copy()

        # Rule-based lookup: most common brand per model
        self.model_to_brand_ = (
            df.dropna(subset=['Brand', 'model'])
              .groupby('model')['Brand']
              .agg(lambda s: s.mode().iloc[0])
        )

        # ML fallback features
        features = [
            'transmission', 'engineSize',
            'fuelType', 'mpg', 'model'
        ]

        # Training data where Brand is known
        train_df = df[df['Brand'].notna()]

        # Train ML model
        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['Brand'])

        # Store trained model and features
        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        #  Rule-based imputation
        # Fill missing Brand values using model to brand lookup
        mask = df['Brand'].isna() & df['model'].notna()
        df.loc[mask, 'Brand'] = df.loc[mask, 'model'].map(self.model_to_brand_)

        # ML fallback
        # Predict remaining missing Brand values
        missing = df['Brand'].isna()
        if missing.any():
            df.loc[missing, 'Brand'] = self.ml_model_.predict(
                df.loc[missing, self.features_]
            )
        return df

class ModelImputer(BaseEstimator, TransformerMixin):
    
    def __init__(self, pipeline_template, min_brand_count=20):
        # Preconfigured ML pipeline for model prediction
        self.pipeline_template = pipeline_template
        # Minimum count threshold for rule-based logic (kept conservative)
        self.min_brand_count = min_brand_count

    def fit(self, X, y=None):
        df = X.copy()

        # Rule-based lookup:
        # Most frequent model for each (Brand, transmission) pair
        self.lookup_ = (
            df.dropna(subset=['Brand', 'transmission', 'model'])
              .groupby(['Brand', 'transmission'])['model']
              .agg(lambda s: s.value_counts().idxmax())
        )

        # ML fallback features
        features = [
            'Brand', 'year', 'engineSize', 'mpg',
            'tax', 'mileage', 'fuelType', 'transmission'
        ]

        # Training data where model is known
        train_df = df[df['model'].notna()]

        # Train ML model
        pipe = clone(self.pipeline_template)
        pipe.fit(train_df[features], train_df['model'])

        # Store trained model and features
        self.ml_model_ = pipe
        self.features_ = features
        return self

    def transform(self, X):
        df = X.copy()

        # Rule-based imputation 
        # Fill missing model values using (Brand, transmission) lookup
        for idx, row in df[df['model'].isna()].iterrows():
            key = (row['Brand'], row['transmission'])
            if key in self.lookup_.index:
                df.at[idx, 'model'] = self.lookup_.loc[key]

        # ML fallback
        # Predict remaining missing model values
        missing = df['model'].isna()
        if missing.any():
            df.loc[missing, 'model'] = self.ml_model_.predict(
                df.loc[missing, self.features_]
            )
        return df


In the next cell we impute the categorical values for train, val and test data. We only use train data to fit the imputer.

In [10]:
# Feature sets used by each imputer
# Each list contains only the columns relevant for predicting the missing variable
# We dont use the price because it's our target

trans_features = ["Brand", "model", "fuelType", "engineSize", "year", "mpg", "tax"]
fuel_features  = ["Brand", "model", "transmission", "engineSize", "year", "mpg"]
brand_features = ["transmission", "engineSize", "fuelType", "mpg", "model"]
model_features = ["Brand", "year", "engineSize", "mpg", "tax", "mileage", "fuelType", "transmission"]

# Create imputer instances with feature-specific pipeline templates
# Each template encodes only the columns that are actually used by the imputer
trans_imputer = TransmissionImputer(make_template_for(trans_features))
fuel_imputer  = FuelTypeImputer(make_template_for(fuel_features))
brand_imputer = BrandImputer(make_template_for(brand_features))
model_imputer = ModelImputer(make_template_for(model_features))

# Fit imputers only on training data to avoid data leakage
for imp in [trans_imputer, fuel_imputer, brand_imputer, model_imputer]:
    imp.fit(x_train)

# Apply the fitted imputers to train, validation, and test sets
# The same learned rules and ML models are reused for all splits
x_train_imp = apply_imputers(x_train)
x_val_imp   = apply_imputers(x_val)
x_test_imp  = apply_imputers(x_test)


The next cell just checks if we successfully removed the categorical missing values, which we did.

In [11]:
missing_report(x_train_imp, "x_train_imp")
missing_report(x_val_imp,   "x_val_imp")
missing_report(x_test_imp,  "x_test_imp")

[x_train_imp] Missing Values (n_rows=63296):


,n_missing,pct_missing
mpg,7653,12.09
tax,6892,10.89
engineSize,1742,2.75
previousOwners,1596,2.52
mileage,1525,2.41
hasDamage,1274,2.01
year,310,0.49


[x_val_imp] Missing Values (n_rows=11171):


,n_missing,pct_missing
mpg,1351,12.09
tax,1216,10.89
engineSize,313,2.80
previousOwners,292,2.61
mileage,276,2.47
hasDamage,247,2.21
year,48,0.43


[x_test_imp] Missing Values (n_rows=32567):


,n_missing,pct_missing
mpg,3840,11.79
tax,3469,10.65
year,1007,3.09
previousOwners,936,2.87
engineSize,875,2.69
mileage,859,2.64
hasDamage,597,1.83


,n_missing,pct_missing
mpg,3840,11.79
tax,3469,10.65
year,1007,3.09
previousOwners,936,2.87
engineSize,875,2.69
mileage,859,2.64
hasDamage,597,1.83


## Intermediate step: Split Brands

After having no missing values in brands left, we can split the data sets into the different brands. In the next cell we check the distribution of brands in our train data.

With those dataframes we build list, so that we can iterate over them.

In [12]:
train_x_list = [x_train_imp]

train_y_list = [y_train]

val_x_list = [x_val_imp]

val_y_list = [y_val]

test_x_list = [x_test_imp]

## 3.2 Numerical Imputation
Strategies for: `mileage`, `year`, `mpg`, `tax`, `engineSize`, `previousOwners`.
These imputers primarily use median values, grouped by relevant features (e.g., median mileage per year) to ensure realistic fills.

In [13]:
class MileageImputer(BaseEstimator, TransformerMixin):
    """
    Imputer for missing mileage values using a hierarchical strategy:
    Fill with median mileage for the car's year
    Fill remaining missing values with the mileage median
    """
    
    def fit(self, X, y=None):
        df = X.copy()
        
        # Compute median mileage per year
        # Used to fill missing values based on the car's year
        self.year_medians_ = df.groupby('year')['mileage'].median()
        
        # Compute median mileage
        self.global_median_ = df['mileage'].median()
        
        return self

    def transform(self, X):
        df = X.copy()
        
        # Fill missing mileage using the median for the corresponding year
        df['mileage'] = df['mileage'].fillna(df['year'].map(self.year_medians_))
        
        # Fill any remaining missing mileage with the mileage median
        df['mileage'] = df['mileage'].fillna(self.global_median_)
        
        return df



class YearImputer(BaseEstimator, TransformerMixin):
    """
    Imputer for missing year values using mileage based binning:
    Fill using median year for mileage bin
    Fill remaining missing values with median year
    """
    def __init__(self, bins=30):
        # Number of bins to divide mileage into
        self.bins = bins

    def fit(self, X, y=None):
        df = X.copy()
        # Bin mileage into specified number of bins
        df['mileage_bin'] = pd.cut(df['mileage'], bins=self.bins)
        # Compute median year per mileage bin
        self.bin_medians_ = df.groupby('mileage_bin')['year'].median()
        # Median as fallback
        self.global_median_ = df['year'].median()
        return self

    def transform(self, X):
        df = X.copy()
        # Bin mileage the same way as in fit
        df['mileage_bin'] = pd.cut(df['mileage'], bins=self.bins)
        # Fill missing year values using mileage-bin median
        df['year'] = df['year'].fillna(df['mileage_bin'].map(self.bin_medians_))
        # Fill remaining missing years with median
        df['year'] = df['year'].fillna(self.global_median_)
        # Remove temporary bin column
        df.drop(columns='mileage_bin', inplace=True)
        return df


class MPGImputer(BaseEstimator, TransformerMixin):
    """
    Imputer for missing mpg using:
    Median per (model, fuelType)
    Median per fuelType
    mpg median
    """
    def fit(self, X, y=None):
        df = X.copy()
        # Median mpg per (model, fuelType)
        self.model_medians_ = df.groupby(['model', 'fuelType'])['mpg'].median()
        # Median mpg per fuelType
        self.fuel_medians_ = df.groupby('fuelType')['mpg'].median()
        # Median mpg
        self.global_median_ = df['mpg'].median()
        return self

    def transform(self, X):
        df = X.copy()
        # Fill missing mpg using (model, fuelType) median
        for idx, row in df[df['mpg'].isna()].iterrows():
            km = (row['model'], row['fuelType'])
            if km in self.model_medians_.index:
                df.at[idx, 'mpg'] = self.model_medians_.loc[km]
        # Fill remaining missing mpg using fuelType median
        df['mpg'] = df['mpg'].fillna(df['fuelType'].map(self.fuel_medians_))
        # Fill remaining missing mpg using global median
        df['mpg'] = df['mpg'].fillna(self.global_median_)
        return df


class PreviousOwnersImputer(BaseEstimator, TransformerMixin):
    """
    Simple imputer for 'previousOwners' using global median
    """
    def fit(self, X, y=None):
        # Store global median number of previous owners
        self.median_ = X['previousOwners'].median()
        return self

    def transform(self, X):
        df = X.copy()
        # Fill missing previousOwners with stored median
        df['previousOwners'] = df['previousOwners'].fillna(self.median_)
        return df


class TaxImputer(BaseEstimator, TransformerMixin):
    """
    Imputer for missing 'tax' values using:
    Median per (model, fuelType, mpg_bin)
    Median per (fuelType, mpg_bin)
    Median per fuelType
    Tax median
    """
    def __init__(self, bins=15):
        # Number of bins to divide mpg into
        self.bins = bins

    def fit(self, X, y=None):
        df = X.copy()
        # Bin mpg into specified number of bins
        df['mpg_bin'] = pd.cut(df['mpg'], bins=self.bins)
        # Median tax per (model, fuelType, mpg_bin)
        self.medians_one_ = df.groupby(['model', 'fuelType', 'mpg_bin'])['tax'].median()
        # Median tax per (fuelType, mpg_bin)
        self.medians_two_ = df.groupby(['fuelType', 'mpg_bin'])['tax'].median()
        # Median tax per fuelType
        self.fuel_medians_ = df.groupby('fuelType')['tax'].median()
        # Median tax
        self.global_median_ = df['tax'].median()
        return self

    def transform(self, X):
        df = X.copy()
        # Bin mpg the same way as in fit
        df['mpg_bin'] = pd.cut(df['mpg'], bins=self.bins)
        # Fill missing tax hierarchically
        for idx, row in df[df['tax'].isna()].iterrows():
            km = (row['model'], row['fuelType'], row['mpg_bin'])
            ke = (row['fuelType'], row['mpg_bin'])
            if km in self.medians_one_.index:
                df.at[idx, 'tax'] = self.medians_one_.loc[km]
            elif ke in self.medians_two_.index:
                df.at[idx, 'tax'] = self.medians_two_.loc[ke]
        # Fill remaining missing tax by fuelType median
        df['tax'] = df['tax'].fillna(df['fuelType'].map(self.fuel_medians_))
        # Fill remaining missing tax by tax median
        df['tax'] = df['tax'].fillna(self.global_median_)
        # Remove temporary bin column
        df.drop(columns='mpg_bin', inplace=True)
        return df


class EngineSizeImputer(BaseEstimator, TransformerMixin):
    """
    Imputer for missing 'engineSize' using:
    Median per (model, fuelType)
    Median per model
    Global median
    """
    def fit(self, X, y=None):
        df = X.copy()
        # Median engineSize per (model, fuelType)
        self.model_medians_ = df.groupby(['model', 'fuelType'])['engineSize'].median()
        # Median engineSize per model only
        self.model_only_medians_ = df.groupby('model')['engineSize'].median()
        # Median engineSize
        self.global_median_ = df['engineSize'].median()
        return self

    def transform(self, X):
        df = X.copy()
        # Fill missing engineSize using (model, fuelType) median
        for idx, row in df[df['engineSize'].isna()].iterrows():
            km = (row['model'], row['fuelType'])
            if km in self.model_medians_.index:
                df.at[idx, 'engineSize'] = self.model_medians_.loc[km]
        # Fill remaining missing engineSize using model-only median
        df['engineSize'] = df['engineSize'].fillna(df['model'].map(self.model_only_medians_))
        # Fill remaining missing engineSize using engineSize median
        df['engineSize'] = df['engineSize'].fillna(self.global_median_)
        return df


Now we create the pipeline to impute the last missing values.

In [14]:
# Each step fills missing values for one specific feature
rule_imputer = Pipeline([
    ("mileage_imp", MileageImputer()),
    ("year_imp", YearImputer()),
    ("mpg_imp", MPGImputer()),
    ("previousOwners_imp", PreviousOwnersImputer()),
    ("tax_imp", TaxImputer()),
    ("engineSize_imp", EngineSizeImputer()),
])

We then train this imputers on train data of the specific brand and then impute the train, val and test data.

In [15]:
for i in range(len(train_x_list)):
    
    # Fit rule-based imputers only on the training data
    # Ensures no data leakage from validation or test sets
    rule_imputer.fit(train_x_list[i])

    # Apply the trained imputers to all splits
    # Sequentially fills missing values in train, validation, and test sets
    train_x_list[i] = rule_imputer.transform(train_x_list[i])
    val_x_list[i]   = rule_imputer.transform(val_x_list[i])
    test_x_list[i]  = rule_imputer.transform(test_x_list[i])


C:\Users\gmbtd\AppData\Local\Temp\ipykernel_14908\2272386602.py:48: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.bin_medians_ = df.groupby('mileage_bin')['year'].median()
C:\Users\gmbtd\AppData\Local\Temp\ipykernel_14908\2272386602.py:130: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  self.medians_one_ = df.groupby(['model', 'fuelType', 'mpg_bin'])['tax'].median()
C:\Users\gmbtd\AppData\Local\Temp\ipykernel_14908\2272386602.py:132: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observ

Because hasDamage only has the value 0 and nan, we set nan to 1 to see if atleast this holds a different value, otherwise it will be useless.

In [16]:
for i in range(len(train_x_list)):
    train_x_list[i]["hasDamage"] = train_x_list[i]["hasDamage"].fillna(1)
    val_x_list[i]["hasDamage"] = val_x_list[i]["hasDamage"].fillna(1)
    test_x_list[i]["hasDamage"] = test_x_list[i]["hasDamage"].fillna(1)

After that we check, if all missing values are gone.

In [17]:
for i in range(len(train_x_list)):
    missing_report(train_x_list[i], f"train_x: {i}")
    missing_report(val_x_list[i],   f"val_x: {i}")
    missing_report(test_x_list[i],  f"test_x: {i}")

[train_x: 0] No missing values found. (n_rows=63296)
[val_x: 0] No missing values found. (n_rows=11171)
[test_x: 0] No missing values found. (n_rows=32567)


## 5. Feature Engineering

For feature enginnering, we try to not create to much new noise for our models, we just add a few features. He we declare the class to use them.

In [18]:
class FeatureEngineeringTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        df = X.copy()
        # Compute median MPG per transmission type
        self.trans_medians_ = df.groupby('transmission')['mpg'].median()
        # Compute 75th and 90th percentiles of mileage for high-mileage flags
        #High-mileage cars often behave differently (lower resale value)
        self.mileage_q75_ = df['mileage'].quantile(0.75)
        self.mileage_q90_ = df['mileage'].quantile(0.90)
        return self

    def transform(self, X):
        df = X.copy()

        # Difference of car's mpg from typical mpg for its transmission
        mpg_median_trans = df['transmission'].map(self.trans_medians_)
        mpg_median_trans = mpg_median_trans.fillna(self.trans_medians_.median())
        df['mpg_diff_transmission'] = (df['mpg'] - mpg_median_trans).fillna(0)

        # Age of the car in years
        df['car_age'] = 2020 - df['year']

        # mpg per unit of engine size
        df['efficiency_ratio'] = df['mpg'] / (df['engineSize'] + 0.1)

        # Average annual mileage
        df['mileage_per_year'] = df['mileage'] / (df['car_age'] + 0.1)

        # Flags for unusually high mileage
        df['high_mileage_flag'] = (df['mileage'] > self.mileage_q75_).astype(int)
        df['very_high_mileage_flag'] = (df['mileage'] > self.mileage_q90_).astype(int)


        return df

Here we apply those features to our dataframes.

In [19]:
for i in range(len(train_x_list)):

    # Fit feature engineering transformer only on training data
    fe = FeatureEngineeringTransformer()
    fe.fit(train_x_list[i])

    # Apply the same transformations to train, validation, and test sets
    train_x_list[i] = fe.transform(train_x_list[i])
    val_x_list[i]   = fe.transform(val_x_list[i])
    test_x_list[i]  = fe.transform(test_x_list[i])


## 6. Scaling

For the models in this notebook, we decided to not use scaling, since it didnt change the score significantly (score was around 1 MAE worse).

## 7. Encoding

For encoding we decided to try to use both Target Encoding and One Hot Encoding next to each other. We also decided to look at the different models for each brand and how many entities we have per model. 

In [20]:
for df in train_x_list:
    print("\n")
    print(df["model"].value_counts())



model
Focus       6007
C-Class     4580
Fiesta      3744
Golf        2977
Corsa       2141
            ... 
A2             1
Z3             1
Veloster       1
StreetKa       1
S5             1
Name: count, Length: 182, dtype: int64


We saw that we have a few models, that dont have that many appearances. To prevent noise and overfitting, we decided to cluster all models under 20 appearances to other. 
A threshold of 20 provided a good balance:
 - It removed extremely rare categories that would otherwise behave like noise,
 - While still preserving the majority of meaningful model groups.

In [21]:
min_count = 20  # Minimum number of occurrences for a model to be kept;lower than this are rare models become "Other"

for i in range(len(train_x_list)):
    # Count occurrences of each car model in training data
    freq = train_x_list[i]["model"].value_counts()
    
    # Identify rare models with fewer than min_count examples
    rare_models = freq[freq < min_count].index
    
    # Replace rare models with "Other" in train, validation, and test sets
    train_x_list[i]["model"] = train_x_list[i]["model"].replace(rare_models, "Other")
    val_x_list[i]["model"]   = val_x_list[i]["model"].replace(rare_models, "Other")
    test_x_list[i]["model"]  = test_x_list[i]["model"].replace(rare_models, "Other")


After that we perform the encoding.

In [22]:
# Columns for One-Hot Encoding (categorical, low-cardinality)
onehot_cols = ['Brand', 'fuelType', 'transmission', 'brand_segment', 'car_segment']

# Columns for Target Encoding (include high-cardinality)
target_cols = ['Brand', 'fuelType', 'transmission', 'model', 'brand_segment', 'car_segment']

# Lists to store encoded DataFrames
train_x_encoded_list = []
val_x_encoded_list   = []
test_x_encoded_list  = []

for i in range(len(train_x_list)):
    X_train_i = train_x_list[i]
    X_val_i   = val_x_list[i]
    X_test_i  = test_x_list[i]
    y_train_i = train_y_list[i]  

    # Keep only columns that exist in this dataset
    onehot_cols_i  = [c for c in onehot_cols if c in X_train_i.columns]
    target_cols_i  = [c for c in target_cols if c in X_train_i.columns]

    # One-Hot Encoder for low-cardinality categorical features
    onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

    # Target Encoder for features correlated with target
    target_encoder = TargetEncoder(target_type="continuous", random_state=42)

    # Combine both encoders in a ColumnTransformer
    encoding_transformer = ColumnTransformer(
        transformers=[
            ('onehot', onehot_encoder, onehot_cols_i),
            ('target', target_encoder, target_cols_i),
        ],
        remainder='passthrough',                # Keep all other columns unchanged
        verbose_feature_names_out=False
    )

    print(f"\n[{i}] fitting encoder on train_x_list[{i}] "
          f"with onehot: {onehot_cols_i} and target-enc: {target_cols_i}")

    # Fit the encoder on training data (important: TargetEncoder uses y)
    encoding_transformer.fit(X_train_i, y_train_i)

    # Transform train, validation, and test sets
    train_arr = encoding_transformer.transform(X_train_i)
    val_arr   = encoding_transformer.transform(X_val_i)
    test_arr  = encoding_transformer.transform(X_test_i)

    # Get output feature names from ColumnTransformer
    feature_names = encoding_transformer.get_feature_names_out()

    # Convert back to DataFrames with original indices
    X_train_enc = pd.DataFrame(train_arr, columns=feature_names, index=X_train_i.index)
    X_val_enc   = pd.DataFrame(val_arr,   columns=feature_names, index=X_val_i.index)
    X_test_enc  = pd.DataFrame(test_arr,  columns=feature_names, index=X_test_i.index)

    # Append encoded DataFrames to the respective lists
    train_x_encoded_list.append(X_train_enc)
    val_x_encoded_list.append(X_val_enc)
    test_x_encoded_list.append(X_test_enc)

print("Encoding (One-Hot + TargetEncoding) for all list entries completed.")



[0] fitting encoder on train_x_list[0] with onehot: ['Brand', 'fuelType', 'transmission'] and target-enc: ['Brand', 'fuelType', 'transmission', 'model']
Encoding (One-Hot + TargetEncoding) for all list entries completed.


We replace our original lists with the encoded ones.

In [23]:
train_x_list = train_x_encoded_list
val_x_list   = val_x_encoded_list
test_x_list  = test_x_encoded_list

<a id="sec-4-filter"></a>
## 8. Feature Selection


For Feature Selection we first use filter methods and then use wrapper methods.

<a id="sec-4-filter"></a>
### 8.1 Filter Methods


We start with Filter methods, because they are computationally inexpensive and helpful for an initial screening of features. 

<a id="sec-4-1-variance"></a>
#### 8.1.1 Variance Threshold (Constant or Quasi-constant)


We therefore first calculate the variance, since values that are always the same dont help us with predicting.

In [24]:
# compute population variance, round for readability, sort by variance desc

train_x_list[0].var(ddof=0).round(6).sort_values(ascending=False)


mileage                   4.623863e+08
mileage_per_year          1.732657e+08
model                     5.237808e+07
transmission              2.971949e+07
Brand                     2.801777e+07
fuelType                  4.448117e+06
tax                       4.010138e+03
efficiency_ratio          1.650681e+02
mpg                       1.295577e+02
mpg_diff_transmission     1.240944e+02
year                      4.573949e+00
car_age                   4.573949e+00
previousOwners            2.027540e+00
engineSize                3.113150e-01
fuelType_Petrol           2.472370e-01
transmission_Manual       2.455320e-01
fuelType_Diesel           2.427540e-01
high_mileage_flag         1.875000e-01
transmission_Semi-Auto    1.774320e-01
Brand_Ford                1.692190e-01
transmission_Automatic    1.614710e-01
Brand_Mercedes-Benz       1.320860e-01
Brand_Volkswagen          1.200990e-01
Brand_Opel                1.099080e-01
very_high_mileage_flag    9.000500e-02
Brand_BMW                

We can see that some Features have a very low variance. We will set a threshold of 0.001 to remove features with variance below this value.

In [25]:
# Remove features with very low variance (threshold = 0.001) as they carry little information
for i in range(len(train_x_list)):
    # Drop low-variance features from training set and get list of dropped columns
    train_x_list[i], dropped_features = drop_low_variance_features(train_x_list[i], threshold=0.001)
    
    # Drop the same features from validation and test sets to keep consistency
    val_x_list[i] = val_x_list[i].drop(columns=dropped_features)
    test_x_list[i] = test_x_list[i].drop(columns=dropped_features)
    #removed features
    print(f"Dropped low-variance features: {list(dropped_features)}")


Dropped low-variance features: ['fuelType_Electric', 'transmission_Other']


<a id="sec-4-2-correlation"></a>
#### 8.1.1 Correlation Analysis
We decided us to not use Correlation Analysis, because it does not help our model.

<a id="sec-5-wrapper"></a>
### 8.2 Wrapper Method: Permutation-based Feature Selection


Instead of Recursive Feature Elimination (RFE), we use a permutation-importance
wrapper method with a Random Forest:

- For each brand, we train a `RandomForestRegressor` on all features of the
   training split.
- We compute permutation importance on the corresponding validation split
   (using R² as the scoring metric). This measures how much the model’s
   performance drops when a feature is randomly shuffled.
- We rank all features by their mean permutation importance and drop the
   bottom 15% (the least important ones).

In [26]:
%%time
Q = 0.15

# Create new lists so the original feature matrices remain unchanged
rf_train_x_list = [None] * len(train_x_list)
rf_val_x_list   = [None] * len(train_x_list)
rf_test_x_list  = [None] * len(train_x_list)

for i in range(len(train_x_list)):

    # Train a baseline RF on all features for this brand
    rf = RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=1,
        max_features="sqrt"
    )
    rf.fit(train_x_list[i], train_y_list[i])

    # Compute permutation importance on the validation set
    perm = permutation_importance(
        rf,
        val_x_list[i],
        val_y_list[i],
        n_repeats=20,
        random_state=42,
        scoring="r2",
        n_jobs=-1
    )

    imp_mean = pd.Series(
        perm.importances_mean,
        index=train_x_list[i].columns
    ).sort_values(ascending=False)

    q15 = imp_mean.quantile(Q)

    # Selection rule: prefer a small positive threshold when importance is noisy
    if q15 > 0.0001:
        threshold = 0.0001
        selected_feats = imp_mean[imp_mean > threshold].index.tolist()
        rule = "drop<=0.0001"
    elif q15 > 0:
        threshold = q15
        selected_feats = imp_mean[imp_mean >= threshold].index.tolist()
        rule = f"quantile@{Q:.2f}"
    else:
        threshold = 0.0
        selected_feats = imp_mean[imp_mean > threshold].index.tolist()
        rule = "drop<=0"

    print(
        f"\nBrand {i}: start={train_x_list[i].shape[1]}  "
        f"kept={len(selected_feats)}  dropped={train_x_list[i].shape[1] - len(selected_feats)}  "
        f"q15={q15:.6f}  rule={rule}  thr={threshold:.6f}"
    )

    # Store filtered matrices in separate lists
    rf_train_x_list[i] = train_x_list[i].loc[:, selected_feats].copy()
    rf_val_x_list[i]   = val_x_list[i].loc[:, selected_feats].copy()
    rf_test_x_list[i]  = test_x_list[i].loc[:, selected_feats].copy()

    print(selected_feats)

# Targets are unchanged
rf_train_y_list = train_y_list
rf_val_y_list   = val_y_list


Brand 0: start=33  kept=30  dropped=3  q15=0.000240  rule=drop<=0.0001  thr=0.000100
['model', 'efficiency_ratio', 'engineSize', 'year', 'car_age', 'mileage', 'Brand', 'transmission', 'mpg', 'transmission_Manual', 'mpg_diff_transmission', 'high_mileage_flag', 'Brand_Mercedes-Benz', 'tax', 'fuelType', 'mileage_per_year', 'fuelType_Petrol', 'fuelType_Diesel', 'Brand_Ford', 'very_high_mileage_flag', 'Brand_Audi', 'Brand_Opel', 'Brand_BMW', 'transmission_Semi-Auto', 'Brand_Volkswagen', 'Brand_Škoda', 'fuelType_Hybrid', 'Brand_Toyota', 'transmission_Automatic', 'Brand_Hyundai']
CPU times: total: 4min 16s
Wall time: 12min 23s


In [28]:
best_params_per_brand_et = [{'bootstrap': False, 'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 1000}]

Q = 0.15

for i in range(len(train_x_list)):

    # train on ALL features (ExtraTrees with tuned params)
    et = ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1,
        **best_params_per_brand_et[i]
    )
    et.fit(train_x_list[i], train_y_list[i])

    #permutation importance on val
    perm = permutation_importance(
        et,
        val_x_list[i],
        val_y_list[i],
        n_repeats=20,
        random_state=42,
        scoring="r2",
        n_jobs=-1
    )

    imp_mean = pd.Series(
        perm.importances_mean,
        index=train_x_list[i].columns
    ).sort_values(ascending=False)

    # 
    # if q15 > 0: do quantile cut (like before)
    #  else: drop everything with importance <= 0
    q15 = imp_mean.quantile(Q)

    if q15 > 0.0001:
        threshold = 0.0001
        selected_feats = imp_mean[imp_mean > threshold].index.tolist()
        dropped_feats  = imp_mean[imp_mean <= threshold].index.tolist()
        rule = "drop<=0.0001"
    elif q15 > 0:
        threshold = q15
        selected_feats = imp_mean[imp_mean >= threshold].index.tolist()
        dropped_feats  = imp_mean[imp_mean <  threshold].index.tolist()
        rule = f"quantile@{Q:.2f}"
    else:
        threshold = 0.0
        selected_feats = imp_mean[imp_mean > threshold].index.tolist()
        dropped_feats  = imp_mean[imp_mean <= threshold].index.tolist()
        rule = "drop<=0"

    print(f"\nAll Brand Rest: start={train_x_list[i].shape[1]}  "
          f"kept={len(selected_feats)}  dropped={len(dropped_feats)}  "
          f"q15={q15:.6f}  rule={rule}  thr={threshold:.6f}")
    

    # update
    train_x_list[i] = train_x_list[i][selected_feats]
    val_x_list[i]   = val_x_list[i][selected_feats]
    test_x_list[i]  = test_x_list[i][selected_feats]
    print(selected_feats)

C:\Users\gmbtd\anaconda3\Lib\site-packages\joblib\externals\loky\process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



All Brand Rest: start=30  kept=30  dropped=0  q15=0.000724  rule=drop<=0.0001  thr=0.000100
['model', 'efficiency_ratio', 'engineSize', 'year', 'car_age', 'transmission_Manual', 'Brand', 'mileage', 'transmission', 'high_mileage_flag', 'mpg', 'mpg_diff_transmission', 'Brand_Mercedes-Benz', 'very_high_mileage_flag', 'Brand_Ford', 'fuelType', 'Brand_Audi', 'fuelType_Petrol', 'tax', 'fuelType_Diesel', 'Brand_Opel', 'Brand_BMW', 'mileage_per_year', 'Brand_Volkswagen', 'fuelType_Hybrid', 'Brand_Toyota', 'transmission_Semi-Auto', 'Brand_Hyundai', 'Brand_Škoda', 'transmission_Automatic']


## 9. Models

For models we tried Random Forest Regressor, Extra Tree Regressor and Hist Gradient Boosting Regressor. For each of those we use grid search and hardcode the result to be able to faster test the models.

### 9.1 Random Forest Regressor

First we use grid search for Random Forest.

['model', 'efficiency_ratio', 'engineSize', 'year', 'car_age', 'transmission_Manual', 'Brand', 'mileage', 'transmission', 'high_mileage_flag', 'mpg', 'mpg_diff_transmission', 'Brand_Mercedes-Benz', 'very_high_mileage_flag', 'Brand_Ford', 'fuelType', 'Brand_Audi', 'fuelType_Petrol', 'tax', 'fuelType_Diesel', 'Brand_Opel', 'Brand_BMW', 'mileage_per_year', 'Brand_Volkswagen', 'fuelType_Hybrid', 'Brand_Toyota', 'transmission_Semi-Auto', 'Brand_Hyundai', 'Brand_Škoda', 'transmission_Automatic']


In [27]:
param_grid = {
    #Number of trees in the forest. More trees more stable predictions and better averaging
    #but slower training and higher memory usage. Too few trees higher variance and less reliable predictions.
    "n_estimators": [200, 300, 400, 500, 600],
    # depth of each tree. higher values  can capture more complex patterns but may overfit.
    #lower values leads to faster training, less overfitting, but may underfit if too low.
    "max_depth": [15, 17, 20, 23, 25],
    #Number of features considered for each split. we use square root or a fraction.
    #Lower values more randomness, reduces overfitting, can increase bias.
    #Higher values more accurate splits, but higher risk of overfitting.
    "max_features": ["sqrt", 0.5],
    #Minimum number of samples required to split a node.
    #Higher values prevents small splits and overfitting may underfit if too large.
    #Lower values allows fine-grained splits, may overfit to noisy data.
    "min_samples_split": [3, 5, 7],
    #Minimum samples required at a leaf node. Higher values → more general predictions and less overfitting.
    #Lower values  more flexible can capture small patterns but may overfit to noise.
    "min_samples_leaf": [1, 2],
}

# Create all possible parameter combinations, create randomness avoid overfit
grid = list(ParameterGrid(param_grid))
n_total = len(grid)

# Lists to store the best parameters and scores per brand
best_params_per_brand_rf = []
best_score_per_brand_rf = []

# Loop over each brand's dataset
for b in range(len(train_x_list)):
    Xtr = rf_train_x_list[b]
    ytr = rf_train_y_list[b]
    Xva = val_x_list[b]
    yva = val_y_list[b]

    best_mae = float("inf")
    best_params = None

    print("\n==============================")
    print(f"Brand {b}: grid search over {n_total} configs")

    # Loop over all parameter combinations
    for i, params in enumerate(grid, start=1):
        print(f"[{i}/{n_total}] params: {params}")

        # Train Random Forest with current parameters
        model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
        model.fit(Xtr, ytr)

        # Predict on validation set and compute MAE
        print(Xva)
        pred = model.predict(Xva)
        mae = mean_absolute_error(yva, pred)
        print(f"    -> MAE: {mae:.3f}")

        # Update best parameters if current MAE is lower
        if mae < best_mae:
            best_mae = mae
            best_params = params
            print(f"    NEW BEST: MAE={best_mae:.3f} | params={best_params}")

    # Store the best parameters and MAE for this brand
    best_params_per_brand_rf.append(best_params)
    best_score_per_brand_rf.append(best_mae)

# Print final best parameters per brand
print("\n===== FINAL BEST PER BRAND =====")
for b, (p, s) in enumerate(zip(best_params_per_brand_rf, best_score_per_brand_rf)):
    print(f"Brand {b}: MAE={s:.3f} | params={p}")


Brand 0: grid search over 300 configs
[1/300] params: {'max_depth': 15, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 3, 'n_estimators': 200}
       Brand_Audi  Brand_BMW  Brand_Ford  Brand_Hyundai  Brand_Mercedes-Benz  \
carID                                                                          
20573         0.0        0.0         1.0            0.0                  0.0   
33499         0.0        0.0         0.0            1.0                  0.0   
36276         0.0        0.0         0.0            0.0                  1.0   
47873         0.0        0.0         0.0            0.0                  0.0   
2484          1.0        0.0         0.0            0.0                  0.0   
...           ...        ...         ...            ...                  ...   
14124         0.0        1.0         0.0            0.0                  0.0   
58275         0.0        0.0         0.0            0.0                  0.0   
9227          0.0        1.0       

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- fuelType_Other
- hasDamage
- previousOwners


Then we use the found parameters for the algorithm to predict the prices of our test data. We therefore also log transform the target.

In [ ]:
# Store the best hyperparameters found for all brands (from grid search)
# Each dictionary contains the parameters that produced the lowest MAE on the validation set
best_params_per_brand_rf = [
    # All Brands (MAE = 1212.034)
    {'max_depth': 20, 'max_features': 0.5,
     'min_samples_leaf': 1, 'min_samples_split': 5,
     'n_estimators': 500}
]

In [56]:
# Lists to store evaluation metrics and test predictions per brand
metrics_per_brand = []
test_pred_list_rf = []
test_pred_list_rf_log = []

for i in range(len(rf_train_x_list)):

    y_train_i = train_y_list[i]
    y_val_i   = val_y_list[i]

    # Ensure y is a Series, not a DataFrame
    if isinstance(y_train_i, pd.DataFrame):
        y_train_i = y_train_i.iloc[:, 0]
    if isinstance(y_val_i, pd.DataFrame):
        y_val_i = y_val_i.iloc[:, 0]

    # Log-transform target for more stable training (reduces skew)
    y_train_log = np.log1p(y_train_i)

    # Train Random Forest on log-transformed target
    model_i = RandomForestRegressor(
        **best_params_per_brand_rf[i], 
        random_state=42,
        n_jobs=-1
    )
    model_i.fit(rf_train_x_list[i], y_train_log)

    # Predict on validation set and revert log-transform
    pred_val_log = model_i.predict(rf_val_x_list[i])
    pred_val = np.expm1(pred_val_log)

    # Compute evaluation metrics
    mae   = mean_absolute_error(y_val_i, pred_val)
    rmse  = np.sqrt(mean_squared_error(y_val_i, pred_val))
    r2    = r2_score(y_val_i, pred_val)
    medae = median_absolute_error(y_val_i, pred_val)

    # Store metrics for this brand
    metrics_per_brand.append({
        "brand_idx": i,  
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "medae": medae,
    })

    # Train final model on combined train + validation set
    x_trainval_i = pd.concat([rf_train_x_list[i], rf_val_x_list[i]], axis=0)
    y_trainval_i = pd.concat([y_train_i, y_val_i], axis=0)

    model_final = RandomForestRegressor(
        **best_params_per_brand_rf[i],
        random_state=42,
        n_jobs=-1
    )
    model_final.fit(x_trainval_i, np.log1p(y_trainval_i))  # log-transform target

    # Predict on test set (log and back-transformed)
    pred_test_log = model_final.predict(rf_test_x_list[i])
    pred_test = np.expm1(pred_test_log)

    # Store predictions per brand
    test_pred_list_rf.append(pd.Series(pred_test, index=rf_test_x_list[i].index, name="price"))
    test_pred_list_rf_log.append(pd.Series(pred_test_log, index=rf_test_x_list[i].index, name="price_log"))

# Convert metrics list to DataFrame for easy inspection
metrics_rf = pd.DataFrame(metrics_per_brand)


NameError: name 'rf_train_x_list' is not defined

After that we build the submission.

In [ ]:
# put all brand predictions back together
test_pred_all = pd.concat(test_pred_list_rf).sort_index()

# build submission
submission = test_pred_all.reset_index()
submission.columns = ["CarID", "price"]
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("total preds:", len(submission))

After that we upload the submission in the submission folder, by the suffix _rf we know which model it was.

In [ ]:
# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}_rf.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


## 9.2 Extra Trees Regressor

For Extra Trees we also use grid search first.

In [ ]:
# Grid of hyperparameters for Extra Trees Regressor
param_grid_et = {
    "n_estimators": [500, 800, 1200],   # Number of trees: more trees more stable predictions, slower training
    "max_depth": [None, 10, 15, 20],    # Max tree depth: None → fully grown; smaller values reduce overfitting
    "max_features": ["sqrt", 0.5],      # Features considered per split: sqrt or fraction
    # Min samples to split a node: higher value less overfitting; low values allow fine-grained splits, which can capture small patterns in the data.
    "min_samples_split": [2, 3, 5],     
    # Min samples per leaf: higher smoother predictions;lower valuecan capture small patterns, may overfit.
    "min_samples_leaf": [1, 2, 3, 5],   
    "bootstrap": [False], # No bootstrap sampling (standard ExtraTrees)
}

# Create all possible parameter combinations, more randomness less overfit
grid = list(ParameterGrid(param_grid_et))
n_total = len(grid)

# Lists to store best hyperparameters and scores per brand
best_params_per_brand_et = []
best_score_per_brand_et = []

# Loop over each brand's dataset
for b in range(len(train_x_list)):
    Xtr = train_x_list[b]
    ytr = train_y_list[b]
    Xva = val_x_list[b]
    yva = val_y_list[b]

    # Ensure y is a Series
    if isinstance(ytr, pd.DataFrame): ytr = ytr.iloc[:, 0]
    if isinstance(yva, pd.DataFrame): yva = yva.iloc[:, 0]

    best_mae = float("inf")
    best_params = None

    print("\n==============================")
    print(f"Brand {b}: ExtraTrees grid search over {n_total} configs")

    # Loop over all parameter combinations
    for i, params in enumerate(grid, start=1):
        print(f"[{i}/{n_total}] params: {params}")

        # Train ExtraTrees on log-transformed target
        model = ExtraTreesRegressor(**params, random_state=42, n_jobs=-1)
        model.fit(Xtr, np.log1p(ytr))

        # Predict on validation set and invert log-transform
        pred_log = model.predict(Xva)
        pred = np.expm1(pred_log)

        # Compute MAE
        mae = mean_absolute_error(yva, pred)
        print(f"    -> MAE (log1p): {mae:.3f}")

        # Update best parameters if current MAE is lower
        if mae < best_mae:
            best_mae = mae
            best_params = params
            print(f"    NEW BEST: MAE={best_mae:.3f} | params={best_params}")

    # Store best hyperparameters and MAE for this brand
    best_params_per_brand_et.append(best_params)
    best_score_per_brand_et.append(best_mae)

# Print final best parameters per brand
print("\n===== FINAL BEST EXTRA-TREES PER BRAND =====")
for b, (p, s) in enumerate(zip(best_params_per_brand_et, best_score_per_brand_et)):
    print(f"Brand {b}: MAE={s:.3f} | params={p}")


Then we use the found parameters for the algorithm to predict the prices of our test data. We therefore also log transform the target.

In [ ]:
# Lists to store predictions and metrics per brand
test_pred_list_et = []
test_pred_list_et_log = []
metrics_per_brand_et = []
val_pred_list_et_log = []

for i in range(len(train_x_list)):
    X_tr, X_va, X_te = train_x_list[i], val_x_list[i], test_x_list[i]
    y_tr, y_va = train_y_list[i], val_y_list[i]

    # Ensure y is a Series
    if isinstance(y_tr, pd.DataFrame): y_tr = y_tr.iloc[:,0]
    if isinstance(y_va, pd.DataFrame): y_va = y_va.iloc[:,0]

    # Use the best hyperparameters for this brand and set criterion to absolute_error
    params_et = best_params_per_brand_et[i].copy()
    params_et['criterion'] = 'absolute_error'

    # --- Train on train set and predict on validation ---
    et_val = ExtraTreesRegressor(**params_et, random_state=42, n_jobs=-1)
    et_val.fit(X_tr, np.log1p(y_tr))  # log-transform target

    pred_va_log = et_val.predict(X_va)
    pred_va = np.expm1(pred_va_log)   # revert log-transform
    val_pred_list_et_log.append(pd.Series(pred_va_log, index=X_va.index, name="price_log"))

    # Compute validation metrics
    mae   = mean_absolute_error(y_va, pred_va)
    rmse  = np.sqrt(mean_squared_error(y_va, pred_va))
    r2    = r2_score(y_va, pred_va)
    medae = median_absolute_error(y_va, pred_va)
    metrics_per_brand_et.append({"brand_idx": i, "mae": mae, "rmse": rmse, "r2": r2, "medae": medae})

    # --- Train on combined train+val and predict test ---
    x_trainval = pd.concat([X_tr, X_va], axis=0)
    y_trainval = pd.concat([y_tr, y_va], axis=0)

    et_final = ExtraTreesRegressor(**params_et, random_state=42, n_jobs=-1)
    et_final.fit(x_trainval, np.log1p(y_trainval))  # log-transform target

    pred_log_te = et_final.predict(X_te)
    pred_te = np.expm1(pred_log_te)  # revert log-transform

    # Store test predictions
    test_pred_list_et.append(pd.Series(pred_te, index=X_te.index, name="price"))
    test_pred_list_et_log.append(pd.Series(pred_log_te, index=X_te.index, name="price_log"))

# Convert metrics list to DataFrame for easy inspection
metrics_et = pd.DataFrame(metrics_per_brand_et)


After that we build the submission.

In [ ]:
# put all brand predictions back together
test_pred_all = pd.concat(test_pred_list_et).sort_index()

# build submission
submission = test_pred_all.reset_index()
submission.columns = ["CarID", "price"]
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("total preds:", len(submission))

After that we upload the submission in the submission folder, by the suffix _etr we know which model it was.

In [ ]:
# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}_etr.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


## 9.3 Hist Gradient Boosting Regressor

In [ ]:
#hyperparameter grid for HGB
param_grid_hgb = {
        # Maximum depth of each tree:
        # lower values simpler trees, less overfitting, may underfit
        # higher values can capture more complex patterns, risk overfitting
    "max_depth": [6, 8],
        # Step size for boosting:
        # lower values slower but more stable training
        # higher values faster but may overshoot optimal predictions
    "learning_rate": [0.03, 0.06, 0.1], 
        # Minimum samples per leaf node:
        # higher values smoother predictions, reduces overfitting
        # lower values are more flexible, may overfit to small patterns/noise
    "min_samples_leaf": [10, 30, 50],
        # L2 regularization on leaf values:
        # higher values stronger regularization, reduces overfitting
        # lower values weaker regularization, more flexible
    "l2_regularization": [0.02, 0.05, 0.1], 
        # Number of bins for continuous features:
        # higher values finer splits, more memory
        # lower values coarser splits, faster training
    "max_bins": [128, 255], 
        # Maximum number of boosting iterations (trees)
    "max_iter": [4000],     
        # Stop training early if validation loss stops improving
    "early_stopping": [True],           
      
}



# Create all parameter combinations
grid = list(ParameterGrid(param_grid_hgb))
n_total = len(grid)

# Store best parameters and MAE per brand
best_params_per_brand_hgb = []
best_score_per_brand_hgb  = []

# Loop over each brand's dataset
for b in range(len(train_x_list)):
    Xtr = train_x_list[b]
    ytr = train_y_list[b]
    Xva = val_x_list[b]
    yva = val_y_list[b]

    # Ensure y is a Series
    if isinstance(ytr, pd.DataFrame): ytr = ytr.iloc[:, 0]
    if isinstance(yva, pd.DataFrame): yva = yva.iloc[:, 0]

    best_mae = float("inf")
    best_params = None

    print("\n==============================")
    print(f"Brand {b}: HGB grid search over {n_total} configs")

    # Loop over all parameter combinations
    for i, params in enumerate(grid, start=1):
        print(f"[{i}/{n_total}] params: {params}")

        # Train HGB regressor
        model = HistGradientBoostingRegressor(**params, random_state=42)
        model.fit(Xtr, np.log1p(ytr))  # log-transform target

        # Predict on validation and revert log-transform
        pred_log = model.predict(Xva)
        pred = np.expm1(pred_log)

        # Compute MAE
        mae = mean_absolute_error(yva, pred)
        print(f"    -> MAE (log1p): {mae:.3f}")

        # Update best parameters if MAE improves
        if mae < best_mae:
            best_mae = mae
            best_params = params
            print(f"    NEW BEST: MAE={best_mae:.3f} | params={best_params}")

    best_params_per_brand_hgb.append(best_params)
    best_score_per_brand_hgb.append(best_mae)

# Print final best parameters per brand
print("\n===== FINAL BEST HGB PER BRAND =====")
for b, (p, s) in enumerate(zip(best_params_per_brand_hgb, best_score_per_brand_hgb)):
    print(f"Brand {b}: MAE={s:.3f} | params={p}")


Then we use the found parameters for the algorithm to predict the prices of our test data. We therefore also log transform the target.

In [ ]:
# Lists to store final predictions and metrics per brand
test_pred_list_hgb = []       # final test predictions (original scale)
test_pred_list_hgb_log = []   # final test predictions (log scale)
metrics_per_brand_hgb = []    # performance metrics per brand
val_pred_list_hgb_log = []    # validation predictions (log scale) for analysis

for i in range(len(train_x_list)):

    # Get training and validation targets
    y_tr = train_y_list[i]
    y_va = val_y_list[i]
    if isinstance(y_tr, pd.DataFrame): y_tr = y_tr.iloc[:, 0]
    if isinstance(y_va, pd.DataFrame): y_va = y_va.iloc[:, 0]

    # Validation predictions
    # Copy best hyperparameters for this brand
    params_hgb = best_params_per_brand_hgb[i].copy()
    params_hgb['loss'] = 'absolute_error'  # Use MAE as the loss function

    # Initialize HistGradientBoostingRegressor with best params
    hgb = HistGradientBoostingRegressor(**params_hgb, random_state=42)
    # Fit on log-transformed target to stabilize variance
    hgb.fit(train_x_list[i], np.log1p(y_tr))

    # Predict on validation set (log scale)
    pred_va_log = hgb.predict(val_x_list[i])
    # Convert predictions back to original scale
    pred_va = np.expm1(pred_va_log)

    # Store validation predictions (log scale)
    val_pred_list_hgb_log.append(
        pd.Series(pred_va_log, index=val_x_list[i].index, name="price_log")
    )

    # Compute metrics on validation set
    mae   = mean_absolute_error(y_va, pred_va)
    rmse  = np.sqrt(mean_squared_error(y_va, pred_va))
    r2    = r2_score(y_va, pred_va)
    medae = median_absolute_error(y_va, pred_va)

    # Store metrics for this brand
    metrics_per_brand_hgb.append({
        "brand_idx": i,  # brand index
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "medae": medae
    })

    print(f"Brand {i}: val MAE={mae:.3f}, RMSE={rmse:.3f}, R2={r2:.3f}, MedAE={medae:.3f}")

    #  train+val -> test predictions
    # Concatenate train + validation sets for final model
    x_trainval = pd.concat([train_x_list[i], val_x_list[i]], axis=0)
    y_trainval = pd.concat([y_tr, y_va], axis=0)

    # Copy best hyperparameters again for final model
    params_hgb_final = best_params_per_brand_hgb[i].copy()
    params_hgb_final['loss'] = 'absolute_error'

    # Initialize and train final model on train+val
    hgb_final = HistGradientBoostingRegressor(**params_hgb_final, random_state=42)
    hgb_final.fit(x_trainval, np.log1p(y_trainval))

    # Predict on test set (log scale)
    pred_te_log = hgb_final.predict(test_x_list[i])
    # Convert predictions back to original scale
    pred_te = np.expm1(pred_te_log)

    # Store test predictions per brand
    test_pred_list_hgb.append(
        pd.Series(pred_te, index=test_x_list[i].index, name="price")
    )
    test_pred_list_hgb_log.append(
        pd.Series(pred_te_log, index=test_x_list[i].index, name="price_log")
    )

# Compile all brand metrics into a single DataFrame
metrics_hgb = pd.DataFrame(metrics_per_brand_hgb)


After that we build the submission.

In [ ]:
# put all brand predictions back together
test_pred_list_hgb = pd.concat(test_pred_list_hgb).sort_index()

# build submission
submission = test_pred_list_hgb.reset_index()
submission.columns = ["CarID", "price"]
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("total preds:", len(submission))

After that we upload the submission in the submission folder, by the suffix _hgbr we know which model it was.

In [ ]:
# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}_hgbr.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


## 9.4 Hybrid Model (Extra Tree Regressor * 0,6 + High Gradient Boosting * 0,4)

For the hybrid model we take the predictions of both models, apply their wheigts and add them toegther. We then create the submission.

In [ ]:
w = 0.6  # weight for blending ET predictions vs HGB predictions
metrics_per_brand_blend = []  # store performance metrics for blended predictions

for i in range(len(val_y_list)):
    # Get validation targets
    y_va = val_y_list[i]
    if isinstance(y_va, pd.DataFrame): 
        y_va = y_va.iloc[:, 0]  # ensure Series

    # Get validation predictions from ET and HGB models (log scale)
    pred_et_va_log  = val_pred_list_et_log[i].copy()
    pred_hgb_va_log = val_pred_list_hgb_log[i].copy()

    # Align indices between true values and predictions
    y_va_s = pd.Series(y_va.values, index=y_va.index)
    y_va_s.index = y_va_s.index.astype(int)

    pred_et_va_log.index  = pred_et_va_log.index.astype(int)
    pred_hgb_va_log.index = pred_hgb_va_log.index.astype(int)

    # Sort all series by index for consistency
    y_va_s = y_va_s.sort_index()
    pred_et_va_log  = pred_et_va_log.sort_index()
    pred_hgb_va_log = pred_hgb_va_log.sort_index()

    # Get common indices across all series
    common_idx = y_va_s.index.intersection(pred_et_va_log.index).intersection(pred_hgb_va_log.index)

    # Blend predictions in log-space and convert back to original scale
    pred_blend_va = np.expm1(
        w * pred_et_va_log.loc[common_idx] + (1 - w) * pred_hgb_va_log.loc[common_idx]
    )

    # True target values for the common indices
    y_true = y_va_s.loc[common_idx]

    # Compute metrics for the blended predictions
    mae   = mean_absolute_error(y_true, pred_blend_va)
    rmse  = np.sqrt(mean_squared_error(y_true, pred_blend_va))
    r2    = r2_score(y_true, pred_blend_va)
    medae = median_absolute_error(y_true, pred_blend_va)

    # Store metrics for this brand
    metrics_per_brand_blend.append({
        "brand_idx": i,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "medae": medae,
    })

# Compile all brand metrics into a DataFrame
metrics_blend = pd.DataFrame(metrics_per_brand_blend)


In [ ]:
test_pred_list_final = []

w = 0.6  # ET-Gewicht in log-space, HGB = 1-w

for i in range(len(test_pred_list_et_log)):
    pred_et_log  = test_pred_list_et_log[i].copy()
    pred_hgb_log = test_pred_list_hgb_log[i].copy()

    # Indizes alignen
    pred_et_log.index  = pred_et_log.index.astype(int)
    pred_hgb_log.index = pred_hgb_log.index.astype(int)

    pred_et_log  = pred_et_log.sort_index()
    pred_hgb_log = pred_hgb_log.sort_index()

    common_idx = pred_et_log.index.intersection(pred_hgb_log.index)

    # log-space blend -> zurück in Originalskala
    pred_final = np.expm1(
        w * pred_et_log.loc[common_idx] + (1 - w) * pred_hgb_log.loc[common_idx]
    )

    test_pred_list_final.append(pd.Series(pred_final, index=common_idx))

test_pred_all = pd.concat(test_pred_list_final).sort_index()

submission = test_pred_all.reset_index()
submission.columns = ["CarID", "price"]

print(submission.head())
print("total preds:", len(submission))
print("n_nans:", test_pred_all.isna().sum())

After that we upload the submission in the submission folder, by the suffix _hybrid we know which model it was.

In [ ]:
# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}_hybrid.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))

# 10. Evaluation

First we check if our models overfit with Random Forest as example.

In [ ]:
test_pred_list_et = []
train_mae_list_et = []
val_mae_list_et = []

train_weights = []   # sample counts per brand (train)
val_weights = []     # sample counts per brand (val)

for i in range(len(train_x_list)):
    X_tr = train_x_list[i]
    X_va = val_x_list[i]
    X_te = test_x_list[i]

    y_tr = train_y_list[i]
    y_va = val_y_list[i]

    # falls y DataFrame -> Series
    if isinstance(y_tr, pd.DataFrame): y_tr = y_tr.iloc[:, 0]
    if isinstance(y_va, pd.DataFrame): y_va = y_va.iloc[:, 0]

    # weights = amount of samples
    train_weights.append(len(y_tr))
    val_weights.append(len(y_va))

    # ====== 1) TRAIN fit -> train + val MAE ======
    et_val = ExtraTreesRegressor(
        **best_params_per_brand_et[i],
        random_state=42,
        n_jobs=-1
    )

    et_val.fit(X_tr, np.log1p(y_tr))

    # train MAE
    pred_tr = np.expm1(et_val.predict(X_tr))
    mae_tr = mean_absolute_error(y_tr, pred_tr)
    train_mae_list_et.append(mae_tr)

    # val MAE
    pred_va = np.expm1(et_val.predict(X_va))
    mae_va = mean_absolute_error(y_va, pred_va)
    val_mae_list_et.append(mae_va)

    print(f"Brand {i}: train MAE = {mae_tr:.3f} | val MAE = {mae_va:.3f}")

    # ====== 2) FINAL fit (train+val) -> test preds ======
    x_trainval = pd.concat([X_tr, X_va], axis=0)
    y_trainval = pd.concat([y_tr, y_va], axis=0)

    et_final = ExtraTreesRegressor(
        **best_params_per_brand_et[i],
        random_state=42,
        n_jobs=-1
    )
    et_final.fit(x_trainval, np.log1p(y_trainval))
    pred_te = np.expm1(et_final.predict(X_te))

    test_pred_list_et.append(pd.Series(pred_te, index=X_te.index, name="price"))

# ===== weighted averages =====
train_weights = np.array(train_weights, dtype=float)
val_weights   = np.array(val_weights, dtype=float)

weighted_train_mae = np.sum(train_weights * np.array(train_mae_list_et)) / np.sum(train_weights)
weighted_val_mae   = np.sum(val_weights   * np.array(val_mae_list_et))   / np.sum(val_weights)

print("\n=== AVERAGES ===")
print(f"Avg train MAE (weighted):    {weighted_train_mae:.3f}")
print(f"Avg val MAE   (weighted):    {weighted_val_mae:.3f}")

After that we want to compare the different models by mae, rmse, r2 and medea. MAE reflects the average error level, RMSE shows how strongly the model is affected by large errors, R² indicates how well the model captures the overall structure of the data, and MedAE is like MAE without being influenced by outliers.

In [ ]:
weights = pd.Series(
    {i: len(val_x_list[i]) for i in range(len(val_x_list))},
    name="weight"
)
def add_weighted_row_metric(df_metric, weights, label="WEIGHTED"):
    df2 = df_metric.copy()
    w = df2["brand_idx"].map(weights).astype(float)

    metric_cols = [c for c in df2.columns if c != "brand_idx"]
    weighted = {"brand_idx": label}
    for c in metric_cols:
        s = df2[c].astype(float)
        mask = s.notna() & w.notna() & (w > 0)
        weighted[c] = (s[mask] * w[mask]).sum() / w[mask].sum()

    return pd.concat([df2, pd.DataFrame([weighted])], ignore_index=True)

In [ ]:
def metric_table(compare_df, metric, weights=None, add_weighted=True):
    dfm = compare_df[pd.to_numeric(compare_df["brand_idx"], errors="coerce").notna()].copy()
    dfm["brand_idx"] = dfm["brand_idx"].astype(int)
    dfm = dfm.sort_values("brand_idx")

    out = dfm[["brand_idx", f"et_{metric}", f"rf_{metric}", f"hgb_{metric}", f"blend_{metric}"]].copy()

    if add_weighted and weights is not None:
        out = add_weighted_row_metric(out, weights, label="WEIGHTED")

    return out

mae_table   = metric_table(compare_df, "mae",   weights)
rmse_table  = metric_table(compare_df, "rmse",  weights)
r2_table    = metric_table(compare_df, "r2",    weights)
medae_table = metric_table(compare_df, "medae", weights)

brand_map = {
    0: "Ford",
    1: "Mercedes-Benz",
    2: "Volkswagen",
    3: "Opel",
    4: "BMW",
    5: "Audi",
    6: "Toyota",
    7: "Škoda",
    8: "Hyundai",
}

for tbl in [mae_table, rmse_table, r2_table, medae_table]:
    tbl["brand_idx"] = tbl["brand_idx"].map(brand_map).fillna(tbl["brand_idx"])
    tbl.rename(columns={"brand_idx": "Brand"}, inplace=True)

for name, tbl in [("mae_table", mae_table),
                  ("rmse_table", rmse_table),
                  ("r2_table", r2_table),
                  ("medae_table", medae_table)]:
    print(f"\n{name}\n" + "-" * len(name))
    print(tbl.to_string(index=False))